In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = 'retina'

import torch
from torchvision.models import resnet50
import torchvision.transforms as T
torch.set_grad_enabled(False)

# COCO classes
CLASSES = [
    'N/A', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A',
    'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse',
    'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack',
    'umbrella', 'N/A', 'N/A', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis',
    'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove',
    'skateboard', 'surfboard', 'tennis racket', 'bottle', 'N/A', 'wine glass',
    'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich',
    'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake',
    'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table', 'N/A',
    'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard',
    'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A',
    'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier',
    'toothbrush'
]


transform = T.Compose([
    T.Resize(800),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# For output bounding box post-processing
def box_cxcywh_to_xyxy(x):
    x_c, y_c, w, h = x.unbind(1)
    b = [(x_c - 0.5 * w), (y_c - 0.5 * h),
         (x_c + 0.5 * w), (y_c + 0.5 * h)]
    return torch.stack(b, dim=1)

def rescale_bboxes(out_bbox, size):
    img_w, img_h = size
    b = box_cxcywh_to_xyxy(out_bbox)
    # Create tensor on the same device as out_bbox to avoid device mismatch
    b = b * torch.tensor([img_w, img_h, img_w, img_h], dtype=torch.float32, device=out_bbox.device)
    return b

def detect(im, model, transform, device, active_heads=None, active_layers=None):
    # Mean-std normalize the input image (batch-size: 1)
    img = transform(im).unsqueeze(0)

    img = img.to(device)
    
    # Propagate through the model
    if active_heads is not None:
        outputs = model(img, effective_heads=active_heads)
    elif active_layers is not None:
        outputs = model(img, active_layers=active_layers)
    else:
        outputs = model(img)
        
    # Keep only predictions with 0.7+ confidence
    probas = outputs['pred_logits'].softmax(-1)[0, :, :-1]
    keep = probas.max(-1).values > 0.7

    # Convert boxes from [0; 1] to image scales
    bboxes_scaled = rescale_bboxes(outputs['pred_boxes'][0, keep], im.size)
    return probas[keep], bboxes_scaled

def predict(im, model, transform, device, active_heads=None, active_layers=None):
    # Mean-std normalize the input image (batch-size: 1)
    img = transform(im).unsqueeze(0)

    img = img.to(device)
    
    # Propagate through the model
    if active_heads is not None:
        outputs = model(img, effective_heads=active_heads)
    elif active_layers is not None:
        outputs = model(img, active_layers=active_layers)
    else:
        outputs = model(img)
        
    probas = outputs['pred_logits'].softmax(-1)[0, :, :-1]
    bboxes_scaled = rescale_bboxes(outputs['pred_boxes'][0], im.size)
    return probas, bboxes_scaled

In [ ]:
import sys
from pathlib import Path
import torch 
# Add project paths
project_root = Path(r'C:\workspace\ml\workspace\master\original')
sys.path.insert(0, str(project_root))

from models import build_model
from evaluation.args import HeadArgs

# Initialize custom DETR model with 4 heads
args = HeadArgs(number_of_heads=4)
model, criterion, postprocessors = build_model(args)

# Load checkpoint
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

checkpoint_path = project_root / 'results' / 'baseline' / '4head' / 'checkpoint.pth'
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['model'])
    print(f"Loaded checkpoint from {checkpoint_path}")
else:
    print(f"Warning: Checkpoint not found at {checkpoint_path}")

model.eval()
print(f"Custom DETR model with {args.nheads} heads loaded on {device}")

In [ ]:
import random
import json
from pathlib import Path
from collections import Counter

# ── Setup COCO dataset ────────────────────────────────────────────────────────
coco_path = 'C:\\workspace\\ml\\data\\coco'
image_set = 'val'
root = Path(coco_path)
assert root.exists(), f'provided COCO path {root} does not exist'

mode = 'instances'
PATHS = {
    "train": (root / "train2017", root / "annotations" / f'{mode}_train2017.json'),
    "val": (root / "val2017", root / "annotations" / f'{mode}_val2017.json'),
}

img_folder, ann_file = PATHS[image_set]

with open(ann_file) as f:
    coco_data = json.load(f)

# ── Select image with selected classes ────────────────────────────────────────
# Find images that contain annotations with selected class indices
category_id_to_idx = {cat['id']: i for i, cat in enumerate(coco_data['categories'])}

valid_image_ids = set()
for ann in coco_data['annotations']:
    cat_idx = category_id_to_idx.get(ann['category_id'])
    if ann['category_id'] in [36]:  # person, car, truck, snowboard
        valid_image_ids.add(ann['image_id'])

if valid_image_ids:
    selected_image_id = random.choice(list(valid_image_ids))
    random_image_info = next(img for img in coco_data['images'] if img['id'] == selected_image_id)
    print(f"Found {len(valid_image_ids)} images with selected classes")
else:
    # Fallback to random image if no images found with selected classes
    print(f"No images found with selected classes, using random image")
    random_image_info = random.choice(coco_data['images'])

file_name = "000000402118.jpg"  # Replace with the desired file name
image_path = img_folder / file_name
im = Image.open(image_path).convert("RGB")

print(f"Selected image: {file_name}")
print(f"Image size: {im.size}")

# ── Run detection ─────────────────────────────────────────────────────────────
scores, boxes = detect(im, model, transform, device)
print(f"Detected {len(scores)} objects with confidence > 0.7")



In [ ]:
import torch
import torch.nn.functional as F
import cv2
import numpy as np

# ── Extract cross-attention weights from last decoder layer using hooks ────────
img_tensor = transform(im).unsqueeze(0).to(device)

# Store attention weights from decoder
cross_attn_weights_list = []

def cross_attn_hook(module, input, output):
    """Hook to capture cross-attention weights"""
    try:
        # Handle different output formats
        if isinstance(output, tuple):
            # Typically: (attn_output, attention_weights)
            if len(output) >= 2:
                attn_weights = output[1]
                if attn_weights is not None and hasattr(attn_weights, 'detach'):
                    cross_attn_weights_list.append(attn_weights.detach())
            elif len(output) == 1:
                # Single element tuple
                if hasattr(output[0], 'detach'):
                    cross_attn_weights_list.append(output[0].detach())
        elif isinstance(output, torch.Tensor):
            # Direct tensor output
            cross_attn_weights_list.append(output.detach())
        else:
            print(f"Warning: Unexpected output type: {type(output)}")
    except Exception as e:
        print(f"Error in hook: {e}")

# Register hook on the last decoder layer's cross-attention
# Access the decoder layers and register hook on cross-attention
decoder_layers = model.transformer.decoder.layers
last_decoder = decoder_layers[-1]

# Register hook on the multihead attention layer
hook_handle = last_decoder.multihead_attn.register_forward_hook(cross_attn_hook)

# Run inference to capture attention weights
with torch.no_grad():
    outputs = model(img_tensor)

# Remove hook
hook_handle.remove()

# Get cross-attention weights from last decoder layer
if cross_attn_weights_list:
    cross_attn_weights = cross_attn_weights_list[-1]
    print(f"Cross-attention weights shape: {cross_attn_weights.shape}")
else:
    print("Warning: Could not capture cross-attention weights")
    cross_attn_weights = None

# Get predicted boxes and logits
pred_logits = outputs['pred_logits']  # (batch_size, num_queries, num_classes+1)
pred_boxes = outputs['pred_boxes']    # (batch_size, num_queries, 4)

# Convert boxes to pixel coordinates
bboxes_pixel = rescale_bboxes(pred_boxes[0], im.size)
probas = pred_logits[0].softmax(-1)

# Filter predictions by confidence threshold
confidence_threshold = 0.7
keep = probas[:, :-1].max(-1).values > confidence_threshold

print(f"\nDetected {keep.sum()} objects with confidence > {confidence_threshold}")
print(f"Image size: {im.size}")

# Get feature map size from cross-attention weights
if cross_attn_weights is not None:
    feat_size = int(np.sqrt(cross_attn_weights.shape[-1]))
    print(f"Feature map size: {feat_size}x{feat_size}")

In [ ]:
import torch
import torch.nn.functional as F
import cv2
import numpy as np

# ── Helper functions for attention visualization ────────────────────────────────
def find_factors(n):
    """Find all factor pairs of n, sorted by how close they are to sqrt(n)"""
    factors = []
    for i in range(1, int(np.sqrt(n)) + 1):
        if n % i == 0:
            factors.append((i, n // i))
    # Sort by difference from square (closest to square first)
    factors.sort(key=lambda x: abs(x[0] - x[1]))
    return factors

def visualize_attention_maps(image, cross_attn_weights, bboxes, probas, 
                             keep_mask, im_size, feat_spatial, num_queries_to_show=6):
    """
    Visualize cross-attention weights for detected objects.
    
    Args:
        image: PIL Image
        cross_attn_weights: Attention weights (batch, num_heads, num_queries, feat_spatial)
        bboxes: Bounding boxes in pixel coordinates
        probas: Class probabilities
        keep_mask: Boolean mask for kept detections
        im_size: Original image size (W, H)
        feat_spatial: Total spatial dimensions of feature map
        num_queries_to_show: Number of top detections to visualize
    """
    img_np = np.array(image)
    img_h, img_w = img_np.shape[:2]
    
    # Convert attention weights to numpy
    attn_weights = cross_attn_weights.cpu().numpy()
    print(f"Attention weights shape: {attn_weights.shape}")
    
    # Handle different possible shapes
    if attn_weights.ndim == 4:
        # Shape: (batch, num_heads, num_queries, feat_spatial)
        attn_weights = attn_weights[0]  # Take first batch
    
    if attn_weights.ndim == 3:
        # Shape: (num_heads, num_queries, feat_spatial)
        attn_weights_avg = attn_weights.mean(axis=0)  # Average across heads
    elif attn_weights.ndim == 2:
        # Shape: (num_queries, feat_spatial)
        attn_weights_avg = attn_weights
    else:
        print(f"Error: Unexpected attention weights shape: {attn_weights.shape}")
        return
    
    print(f"Attention weights (averaged) shape: {attn_weights_avg.shape}")
    
    # Find best factorization of spatial dimensions
    factors = find_factors(feat_spatial)
    feat_h, feat_w = factors[0]  # Get closest to square
    print(f"Feature map shape inferred as: {feat_h}x{feat_w}")
    
    # Get top kept detections
    kept_indices = np.where(keep_mask.cpu().numpy())[0]
    top_indices = kept_indices[:num_queries_to_show]
    
    num_plots = len(top_indices)
    fig, axes = plt.subplots(num_plots, 2, figsize=(12, 5 * num_plots))
    
    if num_plots == 1:
        axes = axes.reshape(1, -1)
    
    for plot_idx, query_idx in enumerate(top_indices):
        # Get attention weights for this query
        attn_map = attn_weights_avg[query_idx]  # (feat_spatial,)
        
        # Reshape attention map to feature dimensions
        attn_map = attn_map.reshape(feat_h, feat_w)
        
        # Normalize to 0-1
        attn_map = (attn_map - attn_map.min()) / (attn_map.max() - attn_map.min() + 1e-8)
        
        # Upsample to original image size using bilinear interpolation
        attn_map_upsampled = cv2.resize(attn_map, (img_w, img_h), interpolation=cv2.INTER_LINEAR)
        
        # Get the predicted class and confidence
        class_probs = probas[query_idx, :-1]  # Exclude background class
        pred_class = class_probs.argmax().item()
        confidence = class_probs.max().item()
        
        # Left plot: Attention heatmap
        ax_left = axes[plot_idx, 0]
        im_display = ax_left.imshow(img_np)
        heatmap = ax_left.imshow(attn_map_upsampled, cmap='jet', alpha=0.5, vmin=0, vmax=1)
        ax_left.set_title(f"Query {query_idx}: {CLASSES[pred_class + 1]}\nConfidence: {confidence:.2%}")
        ax_left.axis('off')
        plt.colorbar(heatmap, ax=ax_left)
        
        # Right plot: Image with bounding box
        ax_right = axes[plot_idx, 1]
        ax_right.imshow(img_np)
        
        # Draw bounding box
        x1, y1, x2, y2 = bboxes[query_idx].cpu().numpy()
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, 
                            edgecolor='green', facecolor='none')
        ax_right.add_patch(rect)
        ax_right.set_title(f"Detected: {CLASSES[pred_class + 1]} ({confidence:.2%})")
        ax_right.axis('off')
    
    plt.tight_layout()
    plt.show()

# ── Extract cross-attention weights and visualize ─────────────────────────────
img_tensor = transform(im).unsqueeze(0).to(device)

# Store attention weights from decoder
cross_attn_weights_list = []

def cross_attn_hook(module, input, output):
    """Hook to capture cross-attention weights"""
    try:
        if isinstance(output, tuple) and len(output) >= 2:
            attn_weights = output[1]
            if attn_weights is not None and hasattr(attn_weights, 'detach'):
                cross_attn_weights_list.append(attn_weights.detach())
        elif isinstance(output, torch.Tensor):
            cross_attn_weights_list.append(output.detach())
    except Exception as e:
        print(f"Error in hook: {e}")

# Register hook on the last decoder layer's cross-attention
decoder_layers = model.transformer.decoder.layers
last_decoder = decoder_layers[-1]
hook_handle = last_decoder.multihead_attn.register_forward_hook(cross_attn_hook)

# Run inference
with torch.no_grad():
    outputs = model(img_tensor)

hook_handle.remove()

# Extract outputs
if cross_attn_weights_list:
    cross_attn_weights = cross_attn_weights_list[-1]
    print(f"Cross-attention weights shape: {cross_attn_weights.shape}")
else:
    print("Warning: Could not capture cross-attention weights")
    cross_attn_weights = None

pred_logits = outputs['pred_logits']
pred_boxes = outputs['pred_boxes']

bboxes_pixel = rescale_bboxes(pred_boxes[0], im.size)
probas = pred_logits[0].softmax(-1)

confidence_threshold = 0.7
keep = probas[:, :-1].max(-1).values > confidence_threshold

print(f"Detected {keep.sum()} objects with confidence > {confidence_threshold}")
print(f"Image size: {im.size}")

if cross_attn_weights is not None:
    # Get spatial dimension from attention weights
    if cross_attn_weights.ndim == 4:
        feat_spatial = cross_attn_weights.shape[-1]
    elif cross_attn_weights.ndim == 3:
        feat_spatial = cross_attn_weights.shape[-1]
    else:
        feat_spatial = 1
    
    print(f"Feature map spatial dimension: {feat_spatial}")
    
    # Visualize attention weights for detected objects
    print(f"\nVisualizing attention weights for top detected objects...")
    visualize_attention_maps(im, cross_attn_weights, bboxes_pixel, probas, 
                            keep, im.size, feat_spatial, num_queries_to_show=6)